This notebook documents the procedure for extracting the relevant data from ensembles of cGENIE Earth System model simulations in the associated study Prow-Fleischer et al. (in submission) 

Because ensembles were run as 5x5 grid spaces with varying pO2 vs PO4, it is set up to iteratively process each ensemble, not all of them simultaneously. Data files available upon request.

# Setup

In [1]:
%%capture
# Captures cell output (must be at the top)

# =======================
# Standard Library Imports
# =======================
import os
import csv
import glob
import re
import string
import sys
import contextlib
import tarfile
import shutil

# =======================
# Data Handling
# =======================
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import netCDF4

# =======================
# Plotting and Visualization
# =======================
import matplotlib.pyplot as plt
from matplotlib import colors as c
from matplotlib.cm import get_cmap

import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# =======================
# Statistics and Modeling
# =======================
from scipy import stats
# import skill_metrics as sm

plt.rcParams['figure.figsize'] = [2, 1]

## Custom Functions

In [2]:
%%capture
#functions

# ======================

# takes all of the *.tar files in a directory, untars them, and saves their contents into a temp folder

def untar_all_to_temp(from_folder, to_folder="temp"):
    os.makedirs(to_folder, exist_ok=True)  # Create temp directory if needed

    all_items = glob.glob(os.path.join(from_folder, "*"))  # Grab everything in from_folder

    for item_path in all_items:
        name = os.path.basename(item_path) #extracts last component of given path

        # Case 1: item is a file
        if os.path.isfile(item_path):
            if tarfile.is_tarfile(item_path):
                print(f"Extracting {name} to {to_folder}")
                try:
                    with tarfile.open(item_path, 'r:*') as tar:
                        tar.extractall(path=to_folder)
                except Exception as e:
                    print(f"Failed to extract {name}: {e}")
            else:
                print(f"Copying file {name} to {to_folder}")
                try:
                    shutil.copy2(item_path, to_folder)
                except Exception as e:
                    print(f"Failed to copy file {name}: {e}")

        # Case 2: item is a directory (not a tar file)
        elif os.path.isdir(item_path):
            print(f"Copying contents of directory {name} to {to_folder}")
            try:
                for root, dirs, files in os.walk(item_path):
                    # relative path inside the directory
                    rel_path = os.path.relpath(root, from_folder)
                    dest_dir = os.path.join(to_folder, rel_path)
        
                    os.makedirs(dest_dir, exist_ok=True)
        
                    for file in files:
                        src_file = os.path.join(root, file)
                        dst_file = os.path.join(dest_dir, file)
                        shutil.copy2(src_file, dst_file)
            except Exception as e:
                print(f"Failed to copy directory {name}: {e}")
            
#=======================
# take each filename in folder, extract either the 2d or 3dnetcdf and save all output into a dataframe sturcutred in two columns with filename and the output file

def tracer2d(df, field):
    df = df.copy()  
    for i in df.index:
        value = df.loc[i, 'content'][field]
        value = value.mean(dim='time')  # use this if one time slice; otherwise select specific time index
        df.at[i, 'content'] = value
    return df

def tracer3d(df, field):
    df = df.copy() 
    for i in df.index:
        value = df.loc[i, 'content'][field]
        value = value.mean(dim='time')  # use this if one time slice; otherwise select specific time index
        df.at[i, 'content'] = value
    return df

#=======================================================================================================================================#
# this takes the timeseries .res file from each folder in file_path directory, reads and saves final value in column_name. Here specifically for extracting SST 

def extract_final_values(file_path, column_name):
    results = []  # List to store extracted values

    for path_to_file in glob.glob(file_path):
        column_names = ["Time (yr)", "Temperature (C)", "_surT (ice-free) (C)", "_benT (C) [> 2000 m]", "_surT (C)"]
       # column_names = ["time (yr)", "global total O2 (mol)",  "global mean O2 (mol kg-1)", "surface (ice-free) O2 (mol kg-1)",  "benthic [> 2000 m] O2 (mol kg-1)",  "surface O2 (mol kg-1)"]
        try:
            series_SST = pd.read_csv(path_to_file, sep ='\s+', names=column_names, skiprows=1)
        except Exception as e:
            print(f"Error reading {path_to_file}: {e}")
            continue  # Skip this file and move to the next one
        
        # Extract the last row's value for the given column
        if column_name in series_SST.columns:
            final_value = series_SST.iloc[-1][column_name]
        else:
            print(f"Warning: Column '{column_name}' not found in {path_to_file}")
            final_value = None  # Assign None if the column is missing

        results.append({"file": os.path.basename(path_to_file), column_name: final_value})

    df_result = pd.DataFrame(results).round(2)

    print(df_result.head())
    return df_result
    
#==========================================

# take the dataframe of site information that holds the proxy data, a list of all ensemble I/Ca simulations, calculate the mscore and plot the values on heatmap
# distinguish between Backgroudn and Kellwasser states

def m_score(plot_me_2d, sites, state, save_path=None, x_labels=None, y_labels=None):
    model_output = {}

    # Extract model values at site locations
    for _, row in plot_me_2d.iterrows():
        fname=row['file']
        model_data = row['content']
        
        model_site = model_data.sel(lat=sites[:, 1], lon=sites[:, 0], method="nearest")
        model_site = model_site.to_numpy()
        

        model_output[fname] = np.diag(model_site)

    proxy_value = sites[:, 2]
    var_proxy = np.var(proxy_value)
    mean_proxy = np.mean(proxy_value)

    mss_array = {}
    for fname, model_vals in model_output.items():
        mse = np.mean(np.square(model_vals - proxy_value))
        var_model = np.var(model_vals)
        mean_model = np.mean(model_vals)

        try:
            mscore = np.abs((2 / np.pi) * np.arcsin(1 - mse / (var_proxy + var_model + np.square(mean_model - mean_proxy))))
        except ValueError:
            mscore = np.nan
        mss_array[fname] = mscore

    mss_values = np.array([mss_array[f] for f in plot_me_2d["file"]])

    if len(mss_values) != len(plot_me_2d): # this warning with print if there are not 25 ensembles in directory
        raise ValueError("Mismatch between number of ensemble entries and M-score results.")

    if y_labels is None or x_labels is None:
        raise ValueError("Both `x_labels` and `y_labels` must be provided for reshaping.")

    # save filenames so they have associated pO2,PO4 and mscore, before reshaping
    df_fnames = pd.DataFrame({"file" : plot_me_2d['file'].values,
    "mscore" : mss_values})  #this is shaped like array
    
    fname_grid = df_fnames["file"].values.reshape(
    (len(y_labels), len(x_labels)))  
    
    df_fnames = pd.DataFrame(
    fname_grid,                                                     #this is shaped like a square
    columns=x_labels, 
    index=y_labels
    )

    df_fnames = df_fnames.sort_index(ascending=False)
    
    mss_grid = mss_values.reshape((len(y_labels), len(x_labels)))
    df_mss = pd.DataFrame(mss_grid, columns=x_labels, index=y_labels) # this is shaped like a square
    df_mss = df_mss.sort_index(ascending=False)

    sns.set()
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(df_mss, cmap="flare", annot=True, fmt=".2f", vmin=0, vmax=0.6)
    ax.set(xlabel="PO4", ylabel="pO2")
    plt.title("M-score for I/Ca")
    plt.suptitle(state)

    if save_path:
        full_path = os.path.join(save_path, f"mscore_{state}.png")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()

    return df_mss, mss_values, df_fnames  #df_mss and df_fnames should be the same grid shape and order, mss_values is an array
        
#=======================================================================================================================================#

# plot from the list of 2D ensemble output mapview of whatever is save into it (here it is assumed that it is I/Ca)
#overlay sites, assumes background I/Ca values

def plot_ensemble_2d_mapview(plot_me_2d, sites, save_path=None, cmap_name='viridis', vmax=4, title=None):
    fig = plt.figure(figsize=[12, 6])
    cmap = plt.get_cmap(cmap_name)

    # Set grid dimensions (5x5 for 25 plots)
    rows, cols, total = 5, 5, 25

    # Subplot arrangement (custom order)
    subplot_order = [20, 21, 22, 23, 24, 15, 16, 17, 18, 19,
                     10, 11, 12, 13, 14, 5, 6, 7, 8, 9,
                     0, 1, 2, 3, 4]

    subplot_kws = dict(projection=ccrs.Robinson(), facecolor='#b0afae')

    # Colorbar axis
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])

    for i, index in enumerate(subplot_order):
        row = plot_me_2d.iloc[index]
        file_name = row['file']
        data_array = row['content']

        ax = plt.subplot(rows, cols, i + 1, **subplot_kws)
        p = data_array.plot(
            x='lon', y='lat', vmin=0, vmax=vmax,
            cmap=cmap, transform=ccrs.PlateCarree(),
            add_labels=False, add_colorbar=False, ax=ax
        )

        # Overlay observation sites
        ax.scatter(
            x=sites[:, 0], y=sites[:, 1], c=sites[:, 2],
            vmin=0, vmax=vmax, cmap=cmap,
            facecolor='none', edgecolor='black',
            marker='o', transform=ccrs.PlateCarree(), s=25
        )

    plt.tight_layout(rect=[0, 0.1, 0.95, 0.97])

    if title:
        plt.suptitle(title, fontsize=16, x=0.5, y=0.98)

    fig.text(0.5, 0.1, 'x$PO_4$', ha='center', fontsize=18)
    fig.text(-0.01, 0.5, '$pO_2$', va='center', rotation='vertical', fontsize=18)

    cb = plt.colorbar(p, cax=cbar_ax, extend="max", orientation='horizontal', label='I/Ca')
    cb.ax.tick_params(labelsize=18)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()

#===============================================

# take from the list of 3d ensemble output, assumed to be dissolved O2, provide a depth and calculate average global dissolved O2 at that depth

def dissolved_O2(plot_me_3d, depth, save_path=None, x_labels=None, y_labels=None):# depth can be an array of depths e.g depth = [0,1,2]
    df3d = {}
    for index, row in plot_me_3d.iterrows():
        O2_conc = row['content']
        depth_mean = O2_conc.isel(zt=depth).mean() 
        df3d[index] = depth_mean

    df3d = pd.DataFrame(df3d.items(), columns=['file', 'dissolved_O2'])

    values = np.array([float(x) for x in df3d['dissolved_O2']])*1E6

    dissolved_O2_matrix = values.reshape(len(y_labels), len(x_labels))
    dissolved_O2_df = pd.DataFrame(dissolved_O2_matrix, columns=x_labels, index=y_labels)
    dissolved_O2_df = dissolved_O2_df.astype(float)

    return dissolved_O2_df, values

#================================================

# take the list of ensemble 2D output, calculate benthic anoxia at a specific threshold dissolved O2 value
    
def benthic_anoxia(df2d, plot_me_2d, anoxia_threshold, save_path=None, x_labels=None, y_labels=None):
    results = {}
    ds = df2d.loc[0, 'content']
    grid_topo = ds['grid_topo'].fillna(0)
    ocn_floor = np.ma.masked_where(grid_topo == 0, grid_topo, copy=True)

    grid_area = ds['grid_area']
    grid_area_ocn = np.ma.masked_where(ocn_floor.mask == True, grid_area, copy=True)

    benthic_anoxia = {}

    for index, row in plot_me_2d.iterrows():
        b_O2 = row['content']
        b_O2 = b_O2.where(b_O2 > 0)
        b_O2 = b_O2.fillna(0)

        seafloor_anoxic_area = np.ma.masked_where(b_O2 > anoxia_threshold, grid_area_ocn, copy=True)
        seafloor_anoxia_abs = np.ma.sum(seafloor_anoxic_area)
        seafloor_anoxia_per = seafloor_anoxia_abs / np.ma.sum(grid_area_ocn) * 100

        benthic_anoxia[index] = seafloor_anoxia_per

    df_anoxia = pd.DataFrame(benthic_anoxia.items(), columns=['file', '%anoxia'])
    
    df_anoxia['%anoxia'] = df_anoxia['%anoxia'].fillna(0)

    # Reshape for heatmap to be consistent with the mscore map
    
    anoxia = df_anoxia['%anoxia'].to_numpy().reshape(len(y_labels), len(x_labels))
    anoxia_df = pd.DataFrame(anoxia, columns=x_labels, index=y_labels)
    anoxia_df = anoxia_df.sort_index(ascending=False)
    anoxia_df = anoxia_df.astype(float)

    # Plot as a heatmap 
    sns.set()
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(anoxia_df, cmap="crest", annot=True, fmt=".1f", vmin=0, vmax=80)
    ax.set(xlabel="$xPO_4$", ylabel="$xO_2$")
    plt.title("0 $\\mu$ mol/mol $O_2$ threshold")
    plt.suptitle("% Seafloor Anoxia")

    if save_path:
        full_path = os.path.join(save_path, "ben_o2.png")
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()


  # take the output and pivot/reshape it to be printed into csv
    anoxia_df = anoxia_df.round(2)
    anoxia_reshape = anoxia_df.reset_index().rename(columns={'index': 'pO2'})
    anoxia_reshape.fillna(0, inplace=True)
    anoxia_reshape = anoxia_reshape.melt(id_vars='pO2', var_name='PO4', value_name='ben_anoxia')

    anoxia_reshape['pO2'] = anoxia_reshape['pO2'].astype(str).str.lstrip('x').astype(float)
    anoxia_reshape['PO4'] = anoxia_reshape['PO4'].astype(str).str.lstrip('x').astype(float)
    
    results['ben_anoxia'] = anoxia_reshape['ben_anoxia'].values

    return results

#================================================================================================================#

#function to take in all extracted data and save as a csv called results.csv in its own directory

def save_results(df_mss, state, pCO2, efolding, globe_avg_temp, sur_O2, path_to_output, output_csv, filename, ben_anoxia=None):

    # Reshape M-score DataFrame
    df_mss_reset = df_mss.reset_index().rename(columns={'index': 'pO2'}) # make df_mss array, should pass ms_values but whatever
    df_fnames_reset = filename.reset_index().rename(columns={'index': 'pO2'})

    
    df_results = df_mss_reset.melt(id_vars='pO2', var_name='PO4', value_name='mscore') #melt unpivots dataframe from wide format to long format
    df_files = df_fnames_reset.melt(id_vars='pO2',var_name='PO4',value_name='file')  
    
    # Add metadata columns
    df_results['State'] = state
    df_results['pCO2'] = pCO2
    df_results['efolding (m)'] = efolding
    df_results['globe_avg_temp (C)'] = globe_avg_temp
    df_results['dissolved_O2_sur (molkg-1x1E6)'] = sur_O2
    df_results['file'] = df_files['file']

    if ben_anoxia is not None:
        df_results['benthic_anoxia'] = ben_anoxia['ben_anoxia'].round(2)
        
    # Clean up pO2 and PO4 for sorting/plotting
    df_results['pO2'] = df_results['pO2'].astype(str).str.lstrip('x').astype(float)
    df_results['PO4'] = df_results['PO4'].astype(str).str.lstrip('x').astype(float)

    
    full_path = os.path.join(path_to_output, output_csv)
    os.makedirs(path_to_output, exist_ok=True)
    print(full_path)
    
    # Save: Append to file if it exists, else create new
    write_header = not os.path.exists(full_path)
    df_results.to_csv(full_path, mode='a', header=write_header, index=False)
    
    return df_results.head()

# Data Import 

In [ ]:
#read in proxy data
sites_bkg = pd.read_csv("../data_files/geochemical/sites_ICa_Bkgd.txt", header = None)
sites_kw = pd.read_csv("../data_files/geochemical/sites_ICa_KW.txt", header = None)

sites_bkg.columns = ['%Long', 'Lat', 'I/Ca', 'label']
sites_kw.columns = ['%Long', 'Lat', 'I/Ca', 'label']

sites_array_bkg = sites_bkg[['long', 'lat', 'I/Ca']].to_numpy()
sites_array_kw = sites_kw[['long', 'lat', 'I/Ca']].to_numpy()


In [ ]:
#set working directory
os.chdir('C:/Users/prowa/cGENIE_processing')

# temp_folder_path = 'C:/Users/prowa/cGENIE_processing/temp'
temp_folder_path = "D:/temp"

if os.path.exists(temp_folder_path):
    try:
        shutil.rmtree(temp_folder_path) #deletes any temp folder before writing
        print(f"Temporary folder '{temp_folder_path}' and its contents deleted successfully.")
    except OSError as e:
        print(f"Error: {e.filename} - {e.strerror}.")
else:
    print(f"Temporary folder '{temp_folder_path}' does not exist.")
    
# #-----------------------------------Section to modify

basename = "Final Runs/Devon370Ma_050xRemin_10xpCO2/e"

sub_dir = "Devon370Ma_050xRemin_10xpCO2f"

globular = r"D:\temp\250109.pO2xPO4_5x5f.dat.*"
#-----------------------------------
untar_all_to_temp("D:/" + basename, temp_folder_path)

path_to_csv =  "C:/Users/prowa/cGENIE_processing/Results/Results_/"+ sub_dir
path_to_figs = "C:/Users/prowa/cGENIE_processing/Figures/Ensembles/Updated/"+ sub_dir


os.makedirs(path_to_figs, exist_ok=True)
os.makedirs(path_to_csv, exist_ok=True)


path_to_time = globular + "/biogem/biogem_series_ocn_temp.res"
path_to_ens = globular +"/biogem/fields_biogem_2d.nc"
path_to_ens_3d = globular +"/biogem/fields_biogem_3d.nc"

output_csv = "results.csv"


In [ ]:
#set x and y labels for the heatmap

run = basename[-1]

if run == 'a':
    y = ['x0.8','x0.9', 'x1.0','x1.1','x1.2'] #O2
    x = ['x0.5','x0.6', 'x0.7','x0.8','x0.9'] #PO4
    
elif run == 'b':

    y = ['x0.8','x0.9', 'x1.0','x1.1','x1.2']
    x = ['x1.0','x1.1', 'x1.2','x1.3','x1.4']
    
elif run == 'c':

    y = ['x0.8','x0.9', 'x1.0','x1.1','x1.2'] #O2
    x = ['x1.5','x1.6', 'x1.7','x1.8','x1.9'] #PO4

elif run == 'd':
    y = ['x0.3','x0.4', 'x0.5','x0.6','x0.7'] #O2
    x = ['x0.5','x0.6', 'x0.7','x0.8','x0.9']

elif run == 'e':
    y = ['x0.3','x0.4', 'x0.5','x0.6','x0.7'] #O2
    x = ['x1.0','x1.1', 'x1.2','x1.3','x1.4'] #PO4

elif run == 'f':
    y = ['x0.3','x0.4', 'x0.5','x0.6','x0.7'] #O2
    x = ['x1.5','x1.6', 'x1.7','x1.8','x1.9'] #PO4

elif run == 'g':
    y = ['x0.8','x0.9', 'x1.0','x1.1','x1.2'] #O2
    x = ['x2.0','x2.1', 'x2.2','x2.3','x2.4'] #PO4

else: #if h
    y = ['x0.3','x0.4', 'x0.5','x0.6','x0.7'] #O2
    x = ['x2.0','x2.1', 'x2.2','x2.3','x2.4'] #PO4

print(x,y)

In [ ]:
%%capture

# Load data
df2d = {}
filename = {}
for name in glob.glob(path_to_ens): 
    filename[name] = path_to_ens
    data = xr.open_dataset(name)
    df2d[name] = data

# Sort and convert to DataFrame
df2d = pd.DataFrame(sorted(df2d.items()), columns=['file', 'content'])
#df2d['file'] = df2d['file'].str.extract(r'dat\.(.*)\\biogem')
df2d['file'] = df2d['file'].str.extract(r'temp\\(.*?)\\biogem')

df3d = {}
filename = {}
for name in glob.glob(path_to_ens_3d): 
    filename[name] = path_to_ens_3d
    data = xr.open_dataset(name)
    df3d[name] = data

# Sort and convert to DataFrame
df3d = pd.DataFrame(sorted(df3d.items()), columns=['file', 'content'])
#df2d['file'] = df2d['file'].str.extract(r'dat\.(.*)\\biogem')
df3d['file'] = df3d['file'].str.extract(r'temp\\(.*?)\\biogem')


# Use tracer2d function to extract specific fields
plot_me_2d_ICa = tracer2d(df2d, 'proxy_sur_ICa')
plot_me_2d_O2 = tracer2d(df2d, 'ocn_ben_O2')
plot_me_3d_O2 = tracer3d(df3d, 'ocn_O2')

file_names = plot_me_2d_ICa['file'].tolist()


# Process and Save Output

In [ ]:
# calculate subsruface global average dissolved O2, set up to capture theoxycline and top part of OMZ (250 m to 660 m)
matrix, sur_O2 = dissolved_O2(plot_me_3d = plot_me_3d_O2, depth = np.array([2,3,4,5]), x_labels=x, y_labels=y)
matrix

In [ ]:
# plot map view of ensemble I/Ca simulations with avg background value overlayed at each site

plot_ensemble_2d_mapview(
    plot_me_2d=plot_me_2d_ICa,
    sites=sites_array_bkg,  # shape (N, 3): long, lat, value
    save_path=f"{path_to_figs}/ICa_bkg_mapview.png",
    title="Ensemble I/Ca Mapview"
)


In [ ]:
#calculate the average sst
SST_ensemble = extract_final_values(path_to_time, column_name = "_surT (C)") # be sure to change the variable name for each set of data you want to save

In [ ]:
#calculate benthic anoxia
results_anoxia = benthic_anoxia(df2d, plot_me_2d_O2, 0, save_path=path_to_figs, x_labels=x, y_labels=y)

In [ ]:
#calculate mscore fits to background average I/Ca data
df_mss, ms_results, df_fnames = m_score(plot_me_2d_ICa, sites_array_bkg, state = "Background", save_path = path_to_figs, x_labels= x, y_labels = y)


In [ ]:
# save_results(df_mss, "Background", pCO2, efolding, SST_ensemble['_surT (C)'], path_to_csv, output_csv, results_anoxia) #don't use x 
# use integrer for pcO2 (e.g "1; 075") and float for remin depth (eg "0.75")

save_results(df_mss, "Background","10", "0.50", SST_ensemble['_surT (C)'], sur_O2, path_to_csv, output_csv, df_fnames, results_anoxia)


In [ ]:
# the mscore fits is the only thing that changes
df_mss_KW, ms_results_KW, df_fnames = m_score(plot_me_2d_ICa, sites_array_kw, state = "Kellwasser", save_path = path_to_figs, x_labels= x, y_labels = y)


In [ ]:
# performing the operations in this order saves both the background and kellwasser results into same csv file. First time save_results is run it is initialize, the second time the data is appended
save_results(df_mss_KW,"Kellwasser", "10", "0.50", SST_ensemble['_surT (C)'], sur_O2, path_to_csv, output_csv, df_fnames, results_anoxia) 